# Parallel Window A/B Decoder Analysis

This notebook analyzes the staggered A/B window schedule for qLDPC circuit-level decoding.

For `n_A=3, n_B=3`, the implemented schedule is:

```text
A1: s1-s2
B1: s2-s4
A2: s4-s6
B2: s6-s8
A3: s8-...
```

The A layer is decoded first to produce boundary variables, then the residual syndrome is updated, then the B layer is decoded.

In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
PYTHON = ROOT / 'SlidingWindowDecoder' / '.conda-gdg' / 'bin' / 'python'
RESULTS = ROOT / 'ParallelWindowDecoder' / 'results'
RESULTS.mkdir(exist_ok=True)
csv_path = RESULTS / 'ab_window_results.csv'
smoke_csv_path = RESULTS / 'smoke_ab_window_results.csv'
print(ROOT)
print(PYTHON)

## Run Experiments

Start with a small run. Increase `--num-shots` after the schedule and diagnostics look right.

In [ ]:
cmd = [
    str(PYTHON),
    str(ROOT / 'ParallelWindowDecoder' / 'run_experiments.py'),
    '--N', '72',
    '--p-list', '0.002,0.003,0.004',
    '--num-repeat', '10',
    '--num-shots', '100',
    '--a-size', '3',
    '--b-width', '3',
    '--osd-order', '10',
    '--max-iter', '200',
    '--decoders', 'global,sliding,parallel,oracle',
    '--out', str(csv_path),
]
print(' '.join(cmd))
# Uncomment to run from the notebook.
# subprocess.run(cmd, check=True)

## Load Results

In [ ]:
def load_rows(path):
    rows = []
    with path.open() as f:
        for row in csv.DictReader(f):
            row['N'] = int(row['N'])
            row['p'] = float(row['p'])
            row['num_repeat'] = int(row['num_repeat'])
            row['num_shots'] = int(row['num_shots'])
            row['elapsed_sec'] = float(row['elapsed_sec'])
            row['flagged'] = int(row['flagged'])
            row['logical_or_flagged'] = int(row['logical_or_flagged'])
            row['ler'] = float(row['ler'])
            row['diagnostics_json'] = json.loads(row['diagnostics']) if row['diagnostics'] else {}
            rows.append(row)
    return rows

path = csv_path if csv_path.exists() else smoke_csv_path
rows = load_rows(path)
print('loaded', len(rows), 'rows from', path)
for row in rows[:8]:
    print(row['decoder'], 'p=', row['p'], 'LER=', row['ler'], 'flagged=', f"{row['flagged']}/{row['num_shots']}")

## Summary Table

In [ ]:
header = ['decoder', 'p', 'shots', 'flagged', 'failed', 'LER', 'elapsed_sec']
print(' | '.join(header))
print(' | '.join(['---'] * len(header)))
for row in sorted(rows, key=lambda r: (r['p'], r['decoder'])):
    print(' | '.join([
        row['decoder'],
        f"{row['p']:.4g}",
        str(row['num_shots']),
        str(row['flagged']),
        str(row['logical_or_flagged']),
        f"{row['ler']:.4g}",
        f"{row['elapsed_sec']:.3f}",
    ]))

## Logical Error Rate

In [ ]:
decoders = sorted({row['decoder'] for row in rows})
plt.figure(figsize=(7, 4))
for decoder in decoders:
    subset = sorted([row for row in rows if row['decoder'] == decoder], key=lambda r: r['p'])
    plt.plot([row['p'] for row in subset], [row['ler'] for row in subset], marker='o', label=decoder)
plt.xlabel('physical error rate p')
plt.ylabel('logical-or-flagged error rate')
plt.yscale('symlog', linthresh=1e-3)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## Runtime

In [ ]:
plt.figure(figsize=(7, 4))
for decoder in decoders:
    subset = sorted([row for row in rows if row['decoder'] == decoder], key=lambda r: r['p'])
    plt.plot([row['p'] for row in subset], [row['elapsed_sec'] for row in subset], marker='o', label=decoder)
plt.xlabel('physical error rate p')
plt.ylabel('elapsed seconds')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## Window Diagnostics

In [ ]:
for row in rows:
    diag = row['diagnostics_json']
    if row['decoder'] in {'parallel', 'oracle'}:
        print(row['decoder'], 'p=', row['p'])
        print('  A:', diag.get('a_window_rows'))
        print('  B:', diag.get('b_window_rows'))
        print('  local flagged:', diag.get('a_flagged_local'), diag.get('b_flagged_local'))

## Notes

- `oracle` uses true sampled bridge faults for A commits, so it verifies the B residual equation rather than measuring decoder performance.
- `parallel` uses decoded A commits. If `oracle` succeeds but `parallel` fails, the A boundary estimate is the bottleneck.
- The current implementation uses the A/B dependency structure, but the local window calls are still executed by Python loops. The experiment data therefore measures algorithmic behavior, not wall-clock parallel speedup.